# Ingesta de datos de temperaturas medias de CONAGUA
Esta libreta de Jupyter se utilizará para la ingesta y procesamiento de datos relacionados al registro de temperaturas medias en distintas estaciones metereológicas distribuidas en todo México. Los datos provienen de la base de datos de la CONAGUA. 

## Ingesta de datos

In [3]:
# Importación de librerías
import os
import time
import requests
import urllib3

# Desactivar advertencias de certificados SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuración general y definición de rutas
base_url = (
    "https://smn.conagua.gob.mx/tools/RESOURCES/com_archivo_datos_resumenes"
)

# Carpeta destino dentro de data/raw/ subiendo desde notebooks/
main_folder = os.path.join("..", "..", "data", "raw", "temperatura")
os.makedirs(main_folder, exist_ok=True)

# Cabeceras completas para autorizar la petición en CONAGUA
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    ),
    "Referer": (
        "https://smn.conagua.gob.mx/es/climatologia/temperaturas-y-lluvias/resumenes-mensuales-de-temperaturas-y-lluvias"
    ),
}

start_year = 2015
end_year = 2021  # range() excluye el final: abarca 2015 a 2020 inclusive


# BUCLES DE DESCARGA (AÑOS Y MESES)

for current_year in range(start_year, end_year):
  year_str = str(current_year)
  print("----------------------------------------")
  print("Descarga del año: " + year_str)
  print("----------------------------------------")

  # Carpeta anual directa: ../data/raw/temperatura_XXXX
  year_folder = os.path.join(main_folder, "temperatura_" + year_str)
  os.makedirs(year_folder, exist_ok=True)

  for month in range(1, 13):
    mes_str = str(month).zfill(2)
    file_name = year_str + mes_str + "010000TMed.csv"
    file_url = base_url.rstrip("/") + "/" + file_name
    local_path = os.path.join(year_folder, file_name)

    # Si el archivo ya existe localmente, se omite
    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
      print(f"Año {year_str} | Mes {mes_str}: Ya existe localmente")
      continue

    try:
      res = requests.get(file_url, headers=headers, verify=False, timeout=20)

      if res.status_code == 200:
        with open(local_path, "wb") as f:
          f.write(res.content)
        print(f"Año {year_str} | Mes {mes_str}: Descargado con éxito")
      else:
        print(
            f"Año {year_str} | Mes {mes_str}: No disponible (Status"
            f" {res.status_code})"
        )

      # Pausa de cortesía para no saturar al servidor
      time.sleep(0.5)

    except Exception as e:
      print(f"Error al descargar {file_name}: {e}")

print("\nDescargas de 2015 a 2020 completadas.")

----------------------------------------
Descarga del año: 2015
----------------------------------------
Año 2015 | Mes 01: Descargado con éxito
Año 2015 | Mes 02: Descargado con éxito
Año 2015 | Mes 03: Descargado con éxito
Año 2015 | Mes 04: Descargado con éxito
Año 2015 | Mes 05: Descargado con éxito
Año 2015 | Mes 06: Descargado con éxito
Año 2015 | Mes 07: Descargado con éxito
Año 2015 | Mes 08: Descargado con éxito
Año 2015 | Mes 09: Descargado con éxito
Año 2015 | Mes 10: Descargado con éxito
Año 2015 | Mes 11: Descargado con éxito
Año 2015 | Mes 12: Descargado con éxito
----------------------------------------
Descarga del año: 2016
----------------------------------------
Año 2016 | Mes 01: Descargado con éxito
Año 2016 | Mes 02: Descargado con éxito
Año 2016 | Mes 03: Descargado con éxito
Año 2016 | Mes 04: Descargado con éxito
Año 2016 | Mes 05: Descargado con éxito
Año 2016 | Mes 06: Descargado con éxito
Año 2016 | Mes 07: Descargado con éxito
Año 2016 | Mes 08: Descargado 

## Procesamiento y Limpieza

In [5]:
import glob
import os
import pandas as pd

# 1. Definir la ruta hacia los datos crudos desde la carpeta 'notebooks'
# Buscamos recursivamente todos los .csv dentro de las carpetas de temperatura
patron = os.path.join("..", "..", "data", "raw", "temperatura", "temperatura_*", "*.csv")

# Definir rutas absolutas con glob.glob
archivos = sorted(glob.glob(patron))
print(f"Total de archivos CSV encontrados: {len(archivos)}")

# Lista que nos servira para a;adir en cada ciclo un archivo csv
lista_nacional = []

# 2. Procesar cada archivo mensual
for ruta in archivos:
  try:
    df_mes = pd.read_csv(ruta, encoding="latin-1")
    df_mes.columns = [c.strip().lower() for c in df_mes.columns] # Limpiar y normalizar nombres de columnas a minúsculas    
    df_mes["archivo_origen"] = os.path.basename(ruta) # Añadir una columna con el nombre del archivo original
    lista_nacional.append(df_mes) # Añadir a la lista el df acctual
  except Exception as e:
    print(f"Error al leer {ruta}: {e}")

  
# 3. Concatenar los df
if lista_nacional:
    df_temperaturas_nacional = pd.concat(lista_nacional, ignore_index=True) # Concatenar todas las listas
    print(
        f"¡Dataset nacional consolidado! Filas: {len(df_temperaturas_nacional):,} |"
        f" Columnas: {df_temperaturas_nacional.shape[1]}")
    
# limpieza y tipificación de temporal del dataframe 
#Cambiar el nombre de la columna temperatura
df_temperaturas_nacional.rename(columns={'tmed':'temp_promedio'}, inplace=True)


# Creacion de columnas Fecha, Año, y Mes
df_temperaturas_nacional['anio'] = df_temperaturas_nacional['archivo_origen'].str[:4].astype(int)
df_temperaturas_nacional['mes'] = df_temperaturas_nacional['archivo_origen'].str[4:6].astype(int)

# Conservar solo el periodo de análisis (2015-2020); en data/raw puede haber otros años
df_temperaturas_nacional = df_temperaturas_nacional[
    df_temperaturas_nacional['anio'].between(start_year, end_year)
].copy()
print(f"Filas tras filtrar "+str(start_year)+"-"+str(end_year)+": {len(df_temperaturas_nacional):,}")

# Crear una columan de fecha con los valores de anio y mes
anio = df_temperaturas_nacional['archivo_origen'].str[:4].astype(str)
mes = df_temperaturas_nacional['archivo_origen'].str[4:6].astype(str)

df_temperaturas_nacional['fecha'] = pd.to_datetime(anio + '-' + mes + '-' + '01')
df_temperaturas_nacional.head()

Total de archivos CSV encontrados: 72
¡Dataset nacional consolidado! Filas: 77,027 | Columnas: 7
Filas tras filtrar 2015-2021: {len(df_temperaturas_nacional):,}


/var/folders/lm/3h75_s2s0536ztzvnj8lr68m0000gn/T/ipykernel_14701/3349171241.py:36: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_temperaturas_nacional.rename(columns={'tmed':'temp_promedio'}, inplace=True)


,lon,lat,clave,edo,est,temp_promedio,archivo_origen,anio,mes,fecha
0,-89.68,20.65,ABAYC,YUC,"Abalá, Yuc.",19.8,201501010000TMed.csv,2015,1,2015-01-01
1,-101.54,20.45,ABSGJ,GTO,"Abasolo, Gto.",17.4,201501010000TMed.csv,2015,1,2015-01-01
2,-98.39,24.07,ABSTP,TAMP,"Abasolo, Tamps.",14.4,201501010000TMed.csv,2015,1,2015-01-01
3,-92.80,16.55,ACACP,CHIS,"Acala, Chis.",21.8,201501010000TMed.csv,2015,1,2015-01-01
4,-99.91,16.85,ACAGR,GRO,"Acapulco, Gro.",26.8,201501010000TMed.csv,2015,1,2015-01-01


In [6]:

# Confirmar el directorio desde donde se ejecuta el notebook
print(os.getcwd())

# Guardar en data/processed, en la raíz del proyecto
ruta_salida = os.path.join(
    "..", "..", "data", "processed",
    "temperaturas_nacional_"+str(start_year)+"_"+str(end_year)+".csv"
)

print(os.path.abspath(ruta_salida))
df_temperaturas_nacional.to_csv(
    ruta_salida, index=False, encoding="utf-8-sig"
)

/Users/benancio/Documents/sonora-health-climate-eda/notebooks/01_ingesta
/Users/benancio/Documents/sonora-health-climate-eda/data/processed/temperaturas_nacional_2015_2021.csv


# Validación de datos 
### Schema basándonos en el diccionario de datos

In [8]:
from datetime import date, datetime
from typing import Optional
import numpy as np
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, field_validator


class TemperaturaEstacionSchema(BaseModel):
  model_config = ConfigDict(extra="ignore")

  # Variables de ubicación geográfica
  lon: Optional[float] = Field(default=None, ge=-180.0, le=180.0)
  lat: Optional[float] = Field(default=None, ge=-90.0, le=90.0)

  # Variables de identificación de estación y estado
  clave: Optional[str] = None
  edo: Optional[str] = None
  est: Optional[str] = None

  # Medición meteorológica (temperatura media mensual en °C)
  temp_promedio: Optional[float] = Field(default=None, ge=-50.0, le=65.0)

  # Metadatos del archivo origen
  archivo_origen: Optional[str] = None

  # Variables temporales derivadas
  anio: Optional[int] = Field(default=None, ge=1900, le=2100)
  mes: Optional[int] = Field(default=None, ge=1, le=12)
  fecha: Optional[date] = None

  # 1. Normalizador universal de nulos y valores centinela del SMN (-99.9, -999)
  @field_validator("*", mode="before")
  @classmethod
  def normalize_missing_and_sentinel_values(cls, value):
    if value is None or pd.isna(value):
      return None

    # Si es texto y representa vacío o nan
    if str(value).strip().lower() in ["", "nan", "none", "null"]:
      return None

    # Detectar códigos centinela comunes de CONAGUA/SMN
    try:
      val_num = float(value)
      if val_num in [-99.9, -999.0, -99.0]:
        return None
    except (ValueError, TypeError):
      pass

    return value

  # 2. Limpieza de campos de texto
  @field_validator(
      "clave", "edo", "est", "archivo_origen", mode="before"
  )
  @classmethod
  def convert_text_fields(cls, value):
    if value is None or pd.isna(value):
      return None
    return str(value).strip()

  # 3. Conversión flexible de fechas
  @field_validator("fecha", mode="before")
  @classmethod
  def parse_date(cls, value):
    if value is None or pd.isna(value):
      return None
    if isinstance(value, pd.Timestamp):
      return value.date()
    if isinstance(value, datetime):
      return value.date()
    if isinstance(value, str):
      return datetime.strptime(value.strip()[:10], "%Y-%m-%d").date()
    return value

### Función para validar los datos descargados

In [9]:
from pydantic import ValidationError
import pandas as pd

def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            temperature = TemperaturaEstacionSchema(**row.to_dict())
            validated_rows.append(temperature.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

# Validar el DataFrame consolidado de temperaturas
df_temperaturas_validadas = validate_df(df_temperaturas_nacional)
df_temperaturas_validadas.head(5)

,lon,lat,clave,edo,est,temp_promedio,archivo_origen,anio,mes,fecha
0,-89.68,20.65,ABAYC,YUC,"Abalá, Yuc.",19.8,201501010000TMed.csv,2015,1,2015-01-01
1,-101.54,20.45,ABSGJ,GTO,"Abasolo, Gto.",17.4,201501010000TMed.csv,2015,1,2015-01-01
2,-98.39,24.07,ABSTP,TAMP,"Abasolo, Tamps.",14.4,201501010000TMed.csv,2015,1,2015-01-01
3,-92.80,16.55,ACACP,CHIS,"Acala, Chis.",21.8,201501010000TMed.csv,2015,1,2015-01-01
4,-99.91,16.85,ACAGR,GRO,"Acapulco, Gro.",26.8,201501010000TMed.csv,2015,1,2015-01-01


In [10]:
# 3. Guardar el archivo final
ruta_salida = os.path.join(
    "..","..", "data", "processed", "temperaturas_nacional_"+str(start_year)+"_"+str(end_year)+".csv"
)
df_temperaturas_validadas.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"Archivo exportado exitosamente a: {ruta_salida}")

Archivo exportado exitosamente a: ../../data/processed/temperaturas_nacional_2015_2021.csv
